# Notebook 5 — Validation Against Test Set (2023–2026)
**Layer:** Quality assurance · **Scope:** Pipeline validation, hallucination audit, RMSE%  
**Inputs:** `hdb_feature_test_20260403.csv` · 25 evaluation queries · pipeline from NB4  
**Outputs:** RMSE% per intent · retrieval recall@20 · hallucination audit report

## 5.1 Install & import dependencies

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from collections import defaultdict

FEATURE_DIR = Path("02_feature_layer/training/outputs")
TEST_CSV    = FEATURE_DIR / "hdb_feature_test_20260412.csv"

# Load test set — NEVER use hdb_feature_train_*.csv for evaluation
# README section 6 explicitly reserves test set for final assessment
df_test = pd.read_csv(TEST_CSV)
print(f"Test set: {df_test.shape[0]:,} rows × {df_test.shape[1]} cols")
print(f"Year range: {df_test.transaction_year.min()} – {df_test.transaction_year.max()}")
print(f"Mean price:   ${df_test.resale_price.mean():,.0f}")
print(f"Median price: ${df_test.resale_price.median():,.0f}")

## 5.2 Decode test set one-hot columns

In [ ]:
TOWN_COLS      = [c for c in df_test.columns if c.startswith("town_")]
FLAT_TYPE_COLS = [c for c in df_test.columns if c.startswith("flat_type_")]
FLAT_MDL_COLS  = [c for c in df_test.columns if c.startswith("flat_model_")]

def decode_onehot(row, cols, prefix):
    for c in cols:
        if row[c] == 1:
            return c.replace(prefix, "")
    return "UNKNOWN"

df_test["town"]       = df_test.apply(lambda r: decode_onehot(r, TOWN_COLS, "town_"), axis=1)
df_test["flat_type"]  = df_test.apply(lambda r: decode_onehot(r, FLAT_TYPE_COLS, "flat_type_"), axis=1)
df_test["flat_model"] = df_test.apply(lambda r: decode_onehot(r, FLAT_MDL_COLS, "flat_model_"), axis=1)

print("Towns in test set:", sorted(df_test["town"].unique()))
print("Flat types:", sorted(df_test["flat_type"].unique()))

## 5.3 Construct 25 evaluation queries — 5 per intent

In [ ]:
# All query values (towns, flat types, price bounds) are drawn
# ONLY from hdb_feature_test_20260403.csv. No invented values.

TEST_MEAN   = int(df_test.resale_price.mean())
TEST_MEDIAN = int(df_test.resale_price.median())
PRICE_MIN   = int(df_test.resale_price.min())
PRICE_MAX   = int(df_test.resale_price.max())
LEASE_MIN   = int(df_test.lease_remaining_years.min())
TOWNS       = df_test["town"].unique().tolist()
FLAT_TYPES  = df_test["flat_type"].unique().tolist()

evaluation_queries = [
    # PRICE_ESTIMATION (5)
    {"intent": "PRICE_ESTIMATION",
     "query": f"How much is a 4-room flat in {TOWNS[0]} worth?",
     "ground_truth_median": float(df_test[(df_test.town==TOWNS[0])&(df_test.room_count==4)].resale_price.median())},
    {"intent": "PRICE_ESTIMATION",
     "query": f"What is the median price of a 3-room flat in {TOWNS[1]}?",
     "ground_truth_median": float(df_test[(df_test.town==TOWNS[1])&(df_test.room_count==3)].resale_price.median())},
    {"intent": "PRICE_ESTIMATION",
     "query": f"Price of executive flat in {TOWNS[2]}?",
     "ground_truth_median": float(df_test[(df_test.town==TOWNS[2])&(df_test.flat_type=="EXECUTIVE")].resale_price.median())},
    {"intent": "PRICE_ESTIMATION",
     "query": f"How much for a 5-room flat in {TOWNS[3]} under ${TEST_MEAN+100000:,}?",
     "ground_truth_median": float(df_test[(df_test.town==TOWNS[3])&(df_test.room_count==5)].resale_price.median())},
    {"intent": "PRICE_ESTIMATION",
     "query": f"Typical price of a 4-room flat in {TOWNS[4]}?",
     "ground_truth_median": float(df_test[(df_test.town==TOWNS[4])&(df_test.room_count==4)].resale_price.median())},

    # NEIGHBOURHOOD (5)
    {"intent": "NEIGHBOURHOOD",
     "query": f"What amenities are near {TOWNS[0]} flats?", "ground_truth_median": None},
    {"intent": "NEIGHBOURHOOD",
     "query": f"How accessible are malls from {TOWNS[1]}?", "ground_truth_median": None},
    {"intent": "NEIGHBOURHOOD",
     "query": f"Is {TOWNS[2]} close to MRT stations?", "ground_truth_median": None},
    {"intent": "NEIGHBOURHOOD",
     "query": f"Are there food courts near {TOWNS[3]}?", "ground_truth_median": None},
    {"intent": "NEIGHBOURHOOD",
     "query": f"Tell me about the accessibility of {TOWNS[4]}", "ground_truth_median": None},

    # SCHOOL_CATCHMENT (5)
    {"intent": "SCHOOL_CATCHMENT",
     "query": f"Which areas in {TOWNS[0]} have the best primary schools?", "ground_truth_median": None},
    {"intent": "SCHOOL_CATCHMENT",
     "query": f"Best towns for primary school quality under ${TEST_MEDIAN:,}?", "ground_truth_median": None},
    {"intent": "SCHOOL_CATCHMENT",
     "query": f"How good are primary schools near {TOWNS[1]} 4-room flats?", "ground_truth_median": None},
    {"intent": "SCHOOL_CATCHMENT",
     "query": f"How many schools are within 1km in {TOWNS[2]}?", "ground_truth_median": None},
    {"intent": "SCHOOL_CATCHMENT",
     "query": f"Primary school access for flats in {TOWNS[3]}?", "ground_truth_median": None},

    # INVESTMENT_TEMPORAL (5)
    {"intent": "INVESTMENT_TEMPORAL",
     "query": f"How have {TOWNS[0]} prices changed from 2023 to 2025?", "ground_truth_median": None},
    {"intent": "INVESTMENT_TEMPORAL",
     "query": "Which town had the highest price growth in 2024?", "ground_truth_median": None},
    {"intent": "INVESTMENT_TEMPORAL",
     "query": f"Price trend for 4-room flats in {TOWNS[1]} since 2023?", "ground_truth_median": None},
    {"intent": "INVESTMENT_TEMPORAL",
     "query": "Compare 2023 vs 2025 median prices by town", "ground_truth_median": None},
    {"intent": "INVESTMENT_TEMPORAL",
     "query": f"Is {TOWNS[2]} appreciating faster than average?", "ground_truth_median": None},

    # LEASE_ADVISORY (5)
    {"intent": "LEASE_ADVISORY",
     "query": f"Should I buy a flat with only {LEASE_MIN+10} years lease?", "ground_truth_median": None},
    {"intent": "LEASE_ADVISORY",
     "query": "How does lease remaining affect HDB flat prices?", "ground_truth_median": None},
    {"intent": "LEASE_ADVISORY",
     "query": "Are flats with <60 years lease much cheaper?", "ground_truth_median": None},
    {"intent": "LEASE_ADVISORY",
     "query": f"Price difference between 70-year and 90-year lease flats in {TOWNS[0]}?", "ground_truth_median": None},
    {"intent": "LEASE_ADVISORY",
     "query": "What is the typical discount for short-lease HDB flats?", "ground_truth_median": None},
]

print(f"Evaluation queries built: {len(evaluation_queries)}")
for intent in ["PRICE_ESTIMATION","NEIGHBOURHOOD","SCHOOL_CATCHMENT","INVESTMENT_TEMPORAL","LEASE_ADVISORY"]:
    count = sum(1 for q in evaluation_queries if q["intent"]==intent)
    print(f"  {intent}: {count} queries")

## 5.4 Run pipeline on all 25 queries & collect results

In [ ]:
# Import run_pipeline from Notebook 4 (or copy it here)
# from nb4_pipeline import run_pipeline

results = []
for eq in evaluation_queries:
    try:
        out = run_pipeline(eq["query"])
        out["expected_intent"]    = eq["intent"]
        out["ground_truth_median"] = eq.get("ground_truth_median")
        results.append(out)
        print(f"[OK] {eq['intent']}: {eq['query'][:55]}")
    except Exception as e:
        print(f"[ERR] {eq['query'][:55]} -> {e}")
        results.append({"query": eq["query"], "error": str(e),
                        "expected_intent": eq["intent"], "ground_truth_median": eq.get("ground_truth_median")})

print(f"\nCompleted: {len(results)} / {len(evaluation_queries)}")

## 5.5 Intent classification accuracy

In [ ]:
correct = 0
for r in results:
    if "error" not in r and r.get("intent") == r.get("expected_intent"):
        correct += 1

accuracy = correct / len(results) * 100
print(f"Intent classification accuracy: {correct}/{len(results)} = {accuracy:.1f}%")

# Per-intent breakdown
from collections import Counter
mismatches = [(r["expected_intent"], r.get("intent","ERROR"))
              for r in results if r.get("intent") != r.get("expected_intent")]
if mismatches:
    print("\nMismatches:")
    for exp, got in mismatches:
        print(f"  Expected {exp} → Got {got}")

## 5.6 RMSE% on price estimation queries

In [ ]:
# Use RMSE% not absolute RMSE — README section 6 guidance
# Test prices are 31% higher than training; absolute RMSE would be misleading.

price_results = [r for r in results
                 if r.get("expected_intent")=="PRICE_ESTIMATION"
                 and r.get("ground_truth_median") is not None
                 and "error" not in r
                 and r["answer"].get("estimate_sgd") is not None]

if price_results:
    errors = []
    for r in price_results:
        est    = r["answer"]["estimate_sgd"]
        actual = r["ground_truth_median"]
        pct_err = abs(est - actual) / actual * 100
        errors.append(pct_err)
        print(f"  Query: {r['query'][:50]}")
        print(f"    Estimated: ${est:,.0f} | Actual median: ${actual:,.0f} | Error: {pct_err:.1f}%")

    rmse_pct = np.sqrt(np.mean(np.array(errors)**2))
    print(f"\nRMSE%: {rmse_pct:.2f}%")
    print(f"Mean absolute % error: {np.mean(errors):.2f}%")
else:
    print("No price estimation results with ground truth available.")

## 5.7 Vector retrieval recall@20

In [ ]:
# For each PRICE_ESTIMATION query, check if the ground-truth town's
# transactions are present in the top-20 retrieved comparables.

recall_scores = []

for r in results:
    if r.get("expected_intent") != "PRICE_ESTIMATION" or "error" in r:
        continue
    expected_town = r["slots"].get("town")
    if not expected_town:
        continue
    # In production: check if vector_hits contains flats from expected_town
    # Here we use the vector_hits count as a proxy
    recall_scores.append(r.get("vector_hits", 0))

if recall_scores:
    print(f"Average top-k hits returned: {np.mean(recall_scores):.1f}")
    print(f"Min: {min(recall_scores)}, Max: {max(recall_scores)}")
    print("Note: Full recall@20 requires comparing retrieved address_keys to test set ground truth.")

## 5.8 Hallucination audit

In [ ]:
# Check every response for strings not derivable from the dataset.
# Fail threshold: 0 hallucinations allowed.

VALID_TOWNS_SET = set([
    "ANG MO KIO","BEDOK","BISHAN","BUKIT BATOK","BUKIT MERAH","BUKIT PANJANG",
    "BUKIT TIMAH","CENTRAL AREA","CHOA CHU KANG","CLEMENTI","GEYLANG","HOUGANG",
    "JURONG EAST","JURONG WEST","KALLANG/WHAMPOA","MARINE PARADE","PASIR RIS",
    "PUNGGOL","QUEENSTOWN","SEMBAWANG","SENGKANG","SERANGOON","TAMPINES",
    "TOA PAYOH","WOODLANDS","YISHUN"
])
VALID_FLAT_TYPES_SET = set(["1 ROOM","2 ROOM","3 ROOM","4 ROOM","5 ROOM","EXECUTIVE","MULTI-GENERATION"])
VALID_FLAT_MODELS_SET = set([
    "Model A","Model A2","Model A-Maisonette","Model C","Model D","Model F",
    "Model H","Model J","New Generation","Simplified","Standard","Apartment",
    "Maisonette","Premium Apartment","Improved","Premium","DBSS",
    "Type S1","Type S2","Multi Generation"
])

# Known-safe numeric bounds from the dataset
PRICE_RANGE = (140000, 1700000)
YEAR_RANGE  = (2015, 2026)
LEASE_RANGE = (39, 98)

hallucination_report = []

for r in results:
    if "error" in r or "answer" not in r:
        continue
    ans = r["answer"]
    flags = []

    # Check narrative for any town-like or MRT-like names not in dataset
    narrative = ans.get("narrative", "")
    for token in re.findall(r"[A-Z][A-Z ]{3,}", narrative):
        token = token.strip()
        if (token not in VALID_TOWNS_SET and
            token not in VALID_FLAT_TYPES_SET and
            token not in VALID_FLAT_MODELS_SET and
            len(token) > 4):
            flags.append(f"Unexpected token in narrative: '{token}'")

    # Check price estimate is within dataset bounds
    est = ans.get("estimate_sgd")
    if est and not (PRICE_RANGE[0] <= est <= PRICE_RANGE[1]):
        flags.append(f"Price estimate ${est:,} outside dataset range ${PRICE_RANGE[0]:,}–${PRICE_RANGE[1]:,}")

    # Check key_factors use only known feature names
    known_features = set([
        "level_mid","lease_remaining_years","floor_area_sqm","room_count",
        "dist_to_mrt_m","orientation_score","dist_to_highway_m",
        "dist_to_foodcourt_m","dist_to_nearest_mall_m","mall_count_3km",
        "mall_weighted_access_3km","dist_to_nearest_school_m","school_count_1km",
        "primary_school_quality_1km_weighted","primary_school_top_quality_1km",
        "primary_school_count_1km","resale_price","transaction_year"
    ])
    for kf in ans.get("key_factors", []):
        fname = kf.get("factor_name","").lower().replace(" ","_")
        if fname and fname not in known_features:
            flags.append(f"Unknown factor name: '{fname}'")

    if flags:
        hallucination_report.append({"query": r["query"], "flags": flags})

print(f"Hallucination audit complete.")
print(f"Queries checked: {len(results)}")
print(f"Queries with flags: {len(hallucination_report)}")

if hallucination_report:
    print("\n--- FLAGGED RESPONSES ---")
    for h in hallucination_report:
        print(f"Query: {h['query'][:60]}")
        for flag in h["flags"]:
            print(f"  FLAG: {flag}")
else:
    print("\nPASS — No hallucinations detected.")

## 5.9 Final validation summary

In [ ]:
total_q   = len(results)
errors_q  = sum(1 for r in results if "error" in r)
ok_q      = total_q - errors_q
halluc    = len(hallucination_report)
intent_ok = sum(1 for r in results if "error" not in r and
                r.get("intent") == r.get("expected_intent"))

print("=" * 55)
print("VALIDATION SUMMARY")
print("=" * 55)
print(f"Total queries run:          {total_q}")
print(f"Pipeline errors:            {errors_q}")
print(f"Successful runs:            {ok_q}")
print(f"Intent accuracy:            {intent_ok}/{ok_q} ({intent_ok/ok_q*100:.1f}%)")
print(f"Hallucinations detected:    {halluc}")
print(f"Hallucination pass:         {'YES' if halluc == 0 else 'NO — fix system prompt'}")
if price_results:
    print(f"Price RMSE%:               {rmse_pct:.2f}%")
print("=" * 55)
print("\nNotebook 5 complete — validation done.")